# Notebook 01 — Exploratory Data Analysis

Goal: understand the Telco Churn dataset before building the model.

In [ ]:
# ── Setup (run this cell first in Colab) ──────────────────────────────────
!pip install pandas numpy matplotlib seaborn -q

# Upload telco_churn.csv via Colab file upload, OR mount Drive:
# from google.colab import drive; drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded ✓')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('telco_churn.csv')   # adjust path if needed
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────
churn_counts = df['Churn'].value_counts()
fig, ax = plt.subplots()
bars = ax.bar(churn_counts.index, churn_counts.values, color=['#34d399','#f87171'], edgecolor='none')
for bar, val in zip(bars, churn_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11)
ax.set_title('Churn Distribution', fontsize=14)
ax.set_xlabel('Churn'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()
print(f'Class imbalance ratio: {churn_counts["No"]/churn_counts["Yes"]:.2f}:1')

In [ ]:
# ── Churn rate by contract type ───────────────────────────────────────────
contract_churn = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean() * 100)
ax = contract_churn.plot(kind='bar', color='#6c63ff', edgecolor='none', rot=0)
ax.set_title('Churn Rate by Contract Type (%)', fontsize=13)
ax.set_ylabel('Churn Rate (%)')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width()/2, p.get_height()+0.5), ha='center')
plt.tight_layout(); plt.show()

In [ ]:
# ── Monthly charges distribution by churn ────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for i, (col, ax) in enumerate(zip(['MonthlyCharges','TotalCharges'], axes)):
    for label, color in [('No','#34d399'),('Yes','#f87171')]:
        ax.hist(df[df['Churn']==label][col].dropna(), bins=30, alpha=0.6, label=label, color=color)
    ax.set_title(f'{col} by Churn')
    ax.legend(title='Churn')
plt.tight_layout(); plt.show()

In [ ]:
# ── Tenure distribution ───────────────────────────────────────────────────
fig, ax = plt.subplots()
for label, color in [('No','#34d399'),('Yes','#f87171')]:
    ax.hist(df[df['Churn']==label]['tenure'], bins=30, alpha=0.6, label=label, color=color)
ax.set_title('Tenure Distribution by Churn')
ax.set_xlabel('Tenure (months)'); ax.legend(title='Churn')
plt.tight_layout(); plt.show()
print('Key insight: most churners leave early (short tenure)')

In [ ]:
# ── Correlation heatmap (numeric columns) ─────────────────────────────────
num_df = df[['tenure','MonthlyCharges','TotalCharges']].copy()
num_df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)
corr = num_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

## Key EDA Findings

1. **Churn rate is ~26%** — class imbalance, use `scale_pos_weight` in XGBoost
2. **Month-to-month customers** churn at 43% vs 3% for 2-year contracts
3. **Short tenure** (< 12 months) = highest risk
4. **High monthly charges** ($70+) correlate with higher churn
5. **Electronic check** payment method has the highest churn rate among payment types

Proceed to `02_feature_engineering.ipynb` →